# Build a Reusable MCP Client

Last time we copy-pasted connection code. Let's wrap it in a small helper so any server can be explored in one line.


In [1]:
# !pip install -q "mcp>=2"

In [2]:
%%writefile server_tools.py
from mcp.server.mcpserver import MCPServer
import datetime

mcp = MCPServer("Utils")


@mcp.tool()
def get_time() -> str:
    """Get the current date and time."""
    return datetime.datetime.now().isoformat(timespec="seconds")


@mcp.tool()
def celsius_to_fahrenheit(c: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return round(c * 9 / 5 + 32, 1)


@mcp.tool()
def reverse(text: str) -> str:
    """Reverse a string."""
    return text[::-1]


if __name__ == "__main__":
    mcp.run()

Overwriting server_tools.py


In [3]:
import os, sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def server_params(script):
    return StdioServerParameters(command=sys.executable, args=[script])


# Jupyter's stderr has no real file descriptor, so we send server logs to devnull
DEVNULL = open(os.devnull, "w")

In [4]:
class MCPClient:
    """Tiny helper: connect to a stdio MCP server, list and call tools."""

    def __init__(self, script):
        self.script = script

    async def _run(self, fn):
        async with stdio_client(server_params(self.script), errlog=DEVNULL) as (r, w):
            async with ClientSession(r, w) as session:
                await session.initialize()
                return await fn(session)

    async def list_tools(self):
        async def fn(s):  # this s represent the session variable inside _run
            return (await s.list_tools()).tools

        return await self._run(fn)

    async def call(self, name, **args):
        async def fn(s):
            res = await s.call_tool(name, args)
            return res.content[0].text

        return await self._run(fn)


client = MCPClient("server_tools.py")

In [7]:
for t in await client.list_tools():
    print(t.name, "->", t.input_schema.get("properties", {}))

get_time -> {}
celsius_to_fahrenheit -> {'c': {'title': 'C', 'type': 'number'}}
reverse -> {'text': {'title': 'Text', 'type': 'string'}}


In [8]:
print(await client.call("get_time"))
print(await client.call("celsius_to_fahrenheit", c=37))
print(await client.call("reverse", text="MCP"))

2026-10-28T22:03:20
98.6
PCM


## Recap

- A client = connect → `initialize()` → list → call
- Tools describe their inputs with a JSON schema
- The same pattern works with _any_ MCP server

**Up next:** let an LLM decide which tool to call.
